# IPN Symbolic V2 — chạy thuần luật trên Google Colab

Notebook này đánh giá revision template sau khi chọn neural C (`C_masked_mean_light_dropout`). Nó chỉ dùng keypoint JSON thô trên Google Drive để dựng template/instance graph và chấm matcher 13 lớp IPN; không tải checkpoint neural, không huấn luyện và không cần GPU. Giữ `RUN_TEST=False` trong vòng authoring này.

In [ ]:
from pathlib import Path

# Mặc định chạy branch đã chứa revision post-C; có thể thay bằng commit SHA để khóa tuyệt đối.
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
REPO_URL = 'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git'
REPO_REF = 'origin/HuyenHoang'
RUN_NAME = 'ipn_symbolic_v2_post_c'
BUILD_SPLITS = ['train', 'val']
RUN_TEST = False
OVERWRITE = False
GRAPH_BATCH_SIZE = 250
EXPECTED_COUNTS = {'train': 2443, 'val': 589, 'test': 1101}

assert len(BUILD_SPLITS) == 2 and set(BUILD_SPLITS) == {'train', 'val'}, 'Workflow này luôn cần đủ train và val.'
assert GRAPH_BATCH_SIZE > 0
assert REPO_REF, 'REPO_REF không được rỗng.'
assert not RUN_TEST or set(BUILD_SPLITS) == {'train', 'val'}, 'Chỉ chạy test sau khi đã dựng train và val.'
ACTIVE_SPLITS = list(BUILD_SPLITS) + (['test'] if RUN_TEST else [])
print({'drive': str(DRIVE_DATA_ROOT), 'ref': REPO_REF, 'splits': ACTIVE_SPLITS, 'test': RUN_TEST})

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Tự phục hồi nếu kernel còn giữ DRIVE_DATA_ROOT cũ hoặc chỉ trỏ tới mount root.
default_data_root = Path('/content/drive/MyDrive/ipn-extract-data')
requested_data_root_value = globals().get('DRIVE_DATA_ROOT')
requested_data_root = Path(requested_data_root_value) if requested_data_root_value else default_data_root
candidate_roots = [requested_data_root, default_data_root, Path('/content/drive/My Drive/ipn-extract-data')]
def is_ipn_data_root(path):
    return path.is_dir() and all((path / name).is_file() for name in ('train.txt', 'val.txt', 'test.txt'))
DRIVE_DATA_ROOT = next((path.resolve() for path in candidate_roots if is_ipn_data_root(path)), None)
assert DRIVE_DATA_ROOT is not None, (
    'Không tìm thấy thư mục IPN chứa train.txt, val.txt, test.txt. ' 
    'Hãy mở Files bên trái Colab, Copy path thư mục dữ liệu và gán lại DRIVE_DATA_ROOT.'
)
print('Đã chọn DRIVE_DATA_ROOT:', DRIVE_DATA_ROOT)
REPO_DIR = Path('/content/ipn_symbolic_v2_repo')
RUN_MODE = 'full_with_test' if RUN_TEST else 'train_val'
LOCAL_BOOTSTRAP = Path('/content') / f'{RUN_NAME}_{RUN_MODE}_bootstrap'
LOCAL_ROOT = LOCAL_BOOTSTRAP
LOCAL_DATA = LOCAL_ROOT / 'data'
LOCAL_KEYPOINT_ROOT = LOCAL_DATA / 'keypoints'
LOCAL_GRAPH_ROOT = LOCAL_DATA / 'instance_graphs'
LOCAL_TEMPLATE_ROOT = LOCAL_DATA / 'template_graphs'
ARTIFACTS_BASE = DRIVE_DATA_ROOT / 'runs' / RUN_NAME / RUN_MODE
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
print('Artifacts sẽ đồng bộ dưới:', ARTIFACTS_BASE)

In [ ]:
import subprocess, sys, json, re, shutil, hashlib, uuid

def run(cmd, cwd=None):
    print('$', ' '.join(map(str, cmd)))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd) if cwd else None, check=True, text=True)

if not REPO_DIR.exists():
    run(['git', 'clone', '--filter=blob:none', REPO_URL, REPO_DIR])
run(['git', '-C', REPO_DIR, 'fetch', '--all', '--tags'])
run(['git', '-C', REPO_DIR, 'checkout', '--detach', REPO_REF])
run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'])
def first_file(*candidates):
    for relative in candidates:
        path = REPO_DIR / relative
        if path.is_file(): return path
    return REPO_DIR / candidates[0]
required = {
    'template_builder': first_file('final/code/src_neurosymbolic/kg/build_template_graph.py', 'scripts/build_template_graph.py'),
    'precompute_v2': first_file('final/code/src_neurosymbolic/precompute_instance_graphs_phase_v2.py', 'scripts/precompute_instance_graphs_phase_v2.py'),
    'validator': first_file('scripts/validate_symbolic_inputs.py'),
    'evaluator': first_file('scripts/eval_ipn_template_matching.py'),
}
assert all(path.is_file() for path in required.values()), {k: str(v) for k, v in required.items() if not v.is_file()}
template_yaml = REPO_DIR / 'configs/ipn/gesture_templates.yaml'
assert template_yaml.is_file() and re.search(r'^version:\s*2\s*$', template_yaml.read_text(encoding='utf-8'), re.M), 'Template YAML không phải V2.'
shutil.copy2(template_yaml, LOCAL_ROOT / 'gesture_templates.yaml')
sys.path.insert(0, str(REPO_DIR / 'final/code'))
sys.path.insert(0, str(REPO_DIR))
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
(LOCAL_ROOT / 'code_revision.txt').write_text(commit + '\n', encoding='utf-8')
print('Đã checkout:', commit)

In [ ]:
# Kiểm tra manifest/leakage; chỉ mở keypoint của ACTIVE_SPLITS để không dùng test khi chỉnh template.
from scripts.validate_symbolic_inputs import subject_from_source
def split_rows(path):
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().rsplit(maxsplit=2)
        if len(parts) == 3:
            rows.append((parts[0], parts[1], int(parts[2])))
    return rows

def locate(source, split):
    text = source.replace('\\', '/')
    parts = [part for part in text.split('/') if part]
    candidates = [Path(text), DRIVE_DATA_ROOT / text, DRIVE_DATA_ROOT / 'keypoints' / text]
    if 'keypoints' in parts:
        candidates.append(DRIVE_DATA_ROOT / 'keypoints' / Path(*parts[parts.index('keypoints') + 1:]))
    if split in parts:
        candidates.append(DRIVE_DATA_ROOT / 'keypoints' / Path(*parts[parts.index(split):]))
    for candidate in candidates:
        if candidate.is_file(): return candidate
    raise FileNotFoundError(f'Không tìm thấy keypoint cho {source}; đã thử: {candidates}')

def frame_count(path):
    payload = json.loads(path.read_text(encoding='utf-8'))
    frames = payload if isinstance(payload, list) else payload.get('frames', [])
    return len(frames)

rows_by_split = {}
subjects_by_split = {}
source_owner = {}
manifest_hashes = {}
original_manifest_dir = LOCAL_ROOT / 'original_manifests'
original_manifest_dir.mkdir(parents=True, exist_ok=True)
for split in ('train', 'val', 'test'):
    split_path = DRIVE_DATA_ROOT / f'{split}.txt'
    assert split_path.is_file(), f'Missing {split_path}'
    manifest_hashes[split] = hashlib.sha256(split_path.read_bytes()).hexdigest()
    shutil.copy2(split_path, original_manifest_dir / f'{split}.txt')
    rows_by_split[split] = split_rows(split_path)
    assert rows_by_split[split], f'{split}.txt rỗng'
    assert len(rows_by_split[split]) == EXPECTED_COUNTS[split], f'{split}: expected {EXPECTED_COUNTS[split]}, found {len(rows_by_split[split])}'
    assert {label_id for _, _, label_id in rows_by_split[split]} == set(range(13)), f'{split} thiếu label ID 0..12'
    subjects = set()
    for source, _, _ in rows_by_split[split]:
        source_key = source.replace('\\', '/')
        assert source_key not in source_owner, f'Duplicate source: {source_key} ({source_owner.get(source_key)} và {split})'
        source_owner[source_key] = split
        subject = subject_from_source(source)
        assert subject is not None, f'Không suy ra subject từ {source}'
        subjects.add(subject)
    subjects_by_split[split] = subjects
    if split in ACTIVE_SPLITS:
        checked = [locate(source, split) for source, _, _ in rows_by_split[split]]
        counts = [frame_count(path) for path in checked]
        assert all(count > 0 for count in counts), f'Keypoint không có frame ở {split}'
        print(split, 'clips=', len(checked), 'frames(min/max)=', min(counts), max(counts))
    else:
        print(split, 'manifest clips=', len(rows_by_split[split]), '(locked; chưa đọc keypoint)')
for left, right in (('train', 'val'), ('train', 'test'), ('val', 'test')):
    overlap = subjects_by_split[left] & subjects_by_split[right]
    assert not overlap, f'Subject leakage {left}/{right}: {sorted(overlap)}'
assert '1CM42_3' not in set().union(*subjects_by_split.values()), 'Subject loại trừ 1CM42_3 vẫn có trong split.'
run_config = {
    'commit': commit, 'template_sha256': hashlib.sha256(template_yaml.read_bytes()).hexdigest(),
    'manifest_sha256': manifest_hashes, 'expected_counts': EXPECTED_COUNTS,
    'active_splits': ACTIVE_SPLITS, 'run_test': RUN_TEST, 'graph_batch_size': GRAPH_BATCH_SIZE,
    'predicate_thresholds': {'hand': 0.12, 'finger': 0.04, 'pinch': 0.08, 'trend_consistency': 0.6},
    'abstention': {'min_best_score': 0.15, 'min_top2_margin': 0.02},
    'event_config': {'max_phase_duration': 16, 'min_phase_duration': 4, 'min_overlap_ratio': 0.4, 'min_overlap_frames': 2},
}
run_config_bytes = json.dumps(run_config, indent=2, sort_keys=True).encode('utf-8')
run_config_hash = hashlib.sha256(run_config_bytes).hexdigest()
attempt_id = run_config_hash[:16] if not OVERWRITE else f'{run_config_hash[:16]}-rebuild-{uuid.uuid4().hex[:8]}'
ARTIFACTS = ARTIFACTS_BASE / attempt_id
# Mọi graph/report local cũng phải thuộc đúng attempt để không tái sử dụng ngầm giữa commit/config.
LOCAL_ROOT = Path('/content') / f'{RUN_NAME}_{RUN_MODE}_{attempt_id}'
LOCAL_DATA = LOCAL_ROOT / 'data'
LOCAL_KEYPOINT_ROOT = LOCAL_DATA / 'keypoints'
LOCAL_GRAPH_ROOT = LOCAL_DATA / 'instance_graphs'
LOCAL_TEMPLATE_ROOT = LOCAL_DATA / 'template_graphs'
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(template_yaml, LOCAL_ROOT / 'gesture_templates.yaml')
(LOCAL_ROOT / 'code_revision.txt').write_text(commit + '\n', encoding='utf-8')
original_manifest_dir = LOCAL_ROOT / 'original_manifests'
original_manifest_dir.mkdir(parents=True, exist_ok=True)
for split in ('train', 'val', 'test'):
    shutil.copy2(DRIVE_DATA_ROOT / f'{split}.txt', original_manifest_dir / f'{split}.txt')
previous_config = ARTIFACTS / 'run_config.json'
if previous_config.is_file() and not OVERWRITE:
    assert previous_config.read_bytes() == run_config_bytes, 'Hash collision/config artifact không khớp.'
(LOCAL_ROOT / 'run_config.json').write_bytes(run_config_bytes)
print('Manifest/subject validation: PASS', manifest_hashes)
print('Attempt bất biến:', ARTIFACTS)

In [ ]:
# Khôi phục graph cũ để resume, rồi copy đúng keypoint cần dùng sang SSD Colab.
assert not (ARTIFACTS / 'completed_run.json').is_file(), 'Attempt này đã hoàn tất; dùng kết quả hiện có hoặc đặt OVERWRITE=True để tạo attempt mới.'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
run(['rsync', '-a', str(LOCAL_ROOT / 'run_config.json'), str(LOCAL_ROOT / 'code_revision.txt'), str(LOCAL_ROOT / 'gesture_templates.yaml'), str(original_manifest_dir), str(ARTIFACTS) + '/'])
previous_graphs = ARTIFACTS / 'data' / 'instance_graphs'
# Khi bật test, có thể seed graph train/val từ một attempt train_val đã hoàn tất và cùng provenance.
if RUN_TEST and not previous_graphs.is_dir() and not OVERWRITE:
    compatibility_keys = ('commit', 'template_sha256', 'manifest_sha256', 'expected_counts', 'predicate_thresholds', 'event_config')
    train_val_base = DRIVE_DATA_ROOT / 'runs' / RUN_NAME / 'train_val'
    for candidate in sorted(train_val_base.glob('*'), reverse=True):
        candidate_config = candidate / 'run_config.json'
        candidate_complete = candidate / 'completed_run.json'
        candidate_graphs = candidate / 'data' / 'instance_graphs'
        if not (candidate_config.is_file() and candidate_complete.is_file() and candidate_graphs.is_dir()):
            continue
        old_config = json.loads(candidate_config.read_text(encoding='utf-8'))
        old_complete = json.loads(candidate_complete.read_text(encoding='utf-8'))
        compatible = all(old_config.get(key) == run_config.get(key) for key in compatibility_keys)
        complete_ok = old_complete.get('status') == 'complete' and old_complete.get('run_config_sha256') == hashlib.sha256(candidate_config.read_bytes()).hexdigest()
        if compatible and complete_ok:
            previous_graphs = candidate_graphs
            print('Tái sử dụng cache train/val đã xác minh từ:', candidate)
            break
if previous_graphs.is_dir() and not OVERWRITE:
    LOCAL_GRAPH_ROOT.mkdir(parents=True, exist_ok=True)
    run(['rsync', '-a', str(previous_graphs) + '/', str(LOCAL_GRAPH_ROOT) + '/'])
    print('Đã khôi phục graph từ run trước để resume.')
for split in ACTIVE_SPLITS:
    dst_split = LOCAL_DATA / f'{split}.txt'
    dst_split.parent.mkdir(parents=True, exist_ok=True)
    local_lines = []
    for source, middle, label_id in rows_by_split[split]:
        source_path = locate(source, split)
        try:
            rel = source_path.relative_to(DRIVE_DATA_ROOT / 'keypoints')
        except ValueError:
            source_parts = source.replace('\\', '/').split('/')
            rel = Path(*source_parts[source_parts.index(split):]) if split in source_parts else Path(split) / source_path.name
        destination = LOCAL_KEYPOINT_ROOT / rel
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, destination)
        local_lines.append(f'{destination.as_posix()} {middle} {label_id}')
    dst_split.write_text('\n'.join(local_lines) + '\n', encoding='utf-8')
print('Đã copy raw keypoints vào', LOCAL_KEYPOINT_ROOT)

In [ ]:
# Dựng 13 template graph; sau đó precompute V2 với resume mặc định (OVERWRITE=False).
template_cmd = [sys.executable, required['template_builder'], '--templates', template_yaml, '--output-dir', LOCAL_TEMPLATE_ROOT, '--pretty']
for split in ('train', 'val', 'test'):
    template_cmd += ['--list-file', DRIVE_DATA_ROOT / f'{split}.txt']
run(template_cmd, cwd=REPO_DIR)
assert len([p for p in LOCAL_TEMPLATE_ROOT.glob('*.json') if p.name != 'index.json']) == 13
precompute = required['precompute_v2']
from src_neurosymbolic.precompute_instance_graphs_phase_v2 import output_path_for_source
drive_graph_root = ARTIFACTS / 'data' / 'instance_graphs'
drive_graph_root.mkdir(parents=True, exist_ok=True)
completed_batches = []
for split in ACTIVE_SPLITS:
    split_lines = (LOCAL_DATA / f'{split}.txt').read_text(encoding='utf-8').splitlines()
    for start in range(0, len(split_lines), GRAPH_BATCH_SIZE):
        stop = min(start + GRAPH_BATCH_SIZE, len(split_lines))
        batch_file = LOCAL_DATA / 'batches' / f'{split}_{start:05d}_{stop:05d}.txt'
        batch_file.parent.mkdir(parents=True, exist_ok=True)
        batch_file.write_text('\n'.join(split_lines[start:stop]) + '\n', encoding='utf-8')
        split_files = {name: LOCAL_DATA / f'{name}.txt' for name in ('train', 'val', 'test')}
        split_files[split] = batch_file
        cmd = [sys.executable, precompute, '--keypoint-root', LOCAL_KEYPOINT_ROOT, '--output-root', LOCAL_GRAPH_ROOT, '--train-file', split_files['train'], '--val-file', split_files['val'], '--test-file', split_files['test'], '--splits', split, '--num-frames', '64', '--max-phase-duration', '16', '--min-phase-duration', '4', '--min-overlap-ratio', '0.4', '--min-overlap-frames', '2', '--hand-motion-threshold', '0.12', '--finger-motion-threshold', '0.04', '--pinch-threshold', '0.08', '--trend-consistency', '0.6']
        if OVERWRITE: cmd.append('--overwrite')
        run(cmd, cwd=REPO_DIR)
        batch_rows = split_rows(batch_file)
        missing_batch = [source for source, _, _ in batch_rows if not output_path_for_source(source, LOCAL_KEYPOINT_ROOT, LOCAL_GRAPH_ROOT, split).is_file()]
        assert not missing_batch, f'{split} batch {start}:{stop} thiếu graph: {missing_batch[:3]}'
        run(['rsync', '-a', str(LOCAL_GRAPH_ROOT) + '/', str(drive_graph_root) + '/'])
        completed_batches.append({'split': split, 'start': start, 'stop': stop})
        progress = {'run_config_sha256': hashlib.sha256((LOCAL_ROOT / 'run_config.json').read_bytes()).hexdigest(), 'completed_batches': completed_batches}
        (LOCAL_ROOT / 'graph_progress.json').write_text(json.dumps(progress, indent=2), encoding='utf-8')
        shutil.copy2(LOCAL_ROOT / 'graph_progress.json', ARTIFACTS / 'graph_progress.json')
        print(f'Checkpointed {split} {stop}/{len(split_lines)} về Drive')
    local_rows = split_rows(LOCAL_DATA / f'{split}.txt')
    missing = [source for source, _, _ in local_rows if not output_path_for_source(source, LOCAL_KEYPOINT_ROOT, LOCAL_GRAPH_ROOT, split).is_file()]
    assert not missing, f'{split} thiếu {len(missing)} graph; ví dụ: {missing[:3]}'
    print(split, 'graphs=', len(rows_by_split[split]))
if RUN_TEST:
    run([sys.executable, required['validator'], '--data-root', LOCAL_DATA, '--keypoint-root', LOCAL_KEYPOINT_ROOT, '--template-dir', LOCAL_TEMPLATE_ROOT, '--instance-graph-root', LOCAL_GRAPH_ROOT, '--train-file', LOCAL_DATA / 'train.txt', '--val-file', LOCAL_DATA / 'val.txt', '--test-file', LOCAL_DATA / 'test.txt', '--expected-template-count', '13', '--json-report', LOCAL_ROOT / 'validation.json'], cwd=REPO_DIR)
else:
    print('Đã kiểm tra đủ graph train/val. Validator full-cohort sẽ chạy khi RUN_TEST=True.')

In [ ]:
# Chấm train và val, lưu JSON có accuracy, macro-F1, ties, confusion matrix, per-class.
for split in ('train', 'val'):
    output = LOCAL_ROOT / f'{split}_evaluation.json'
    eval_cmd = [sys.executable, required['evaluator'], '--data-root', LOCAL_DATA, '--split', split, '--split-file', LOCAL_DATA / f'{split}.txt', '--keypoint-root', LOCAL_KEYPOINT_ROOT, '--template-dir', LOCAL_TEMPLATE_ROOT, '--instance-graph-root', LOCAL_GRAPH_ROOT, '--min-best-score', '0.15', '--min-top2-margin', '0.02', '--output', output]
    if split == 'val':
        eval_cmd += ['--diagnostics-jsonl', LOCAL_ROOT / 'val_diagnostics.jsonl']
    run(eval_cmd, cwd=REPO_DIR)
    shutil.copy2(output, ARTIFACTS / output.name)
    if split == 'val': shutil.copy2(LOCAL_ROOT / 'val_diagnostics.jsonl', ARTIFACTS / 'val_diagnostics.jsonl')
    report = json.loads(output.read_text(encoding='utf-8'))
    print(split, {'accuracy': report['accuracy'], 'macro_f1': report['macro_f1'], 'coverage': report['coverage'], 'selective_accuracy': report['selective_accuracy'], 'abstained': report['abstained_count'], 'ties': report['tie_count'], 'clips': report['clips']})
    print('confusion_matrix=', report['confusion_matrix'])
    print('per_class=', report['per_class'])

In [ ]:
# Ô khóa: mặc định không chạy. Chỉ chạy khi RUN_TEST=True và train/val đã thành công.
if RUN_TEST:
    output = LOCAL_ROOT / 'test_evaluation.json'
    run([sys.executable, required['evaluator'], '--data-root', LOCAL_DATA, '--split', 'test', '--split-file', LOCAL_DATA / 'test.txt', '--keypoint-root', LOCAL_KEYPOINT_ROOT, '--template-dir', LOCAL_TEMPLATE_ROOT, '--instance-graph-root', LOCAL_GRAPH_ROOT, '--min-best-score', '0.15', '--min-top2-margin', '0.02', '--output', output], cwd=REPO_DIR)
    shutil.copy2(output, ARTIFACTS / output.name)
    print(json.loads(output.read_text(encoding='utf-8')))
else:
    print('Bỏ qua test vì RUN_TEST=False.')

In [ ]:
# Chỉ đánh dấu hoàn tất sau khi đủ report; không đồng bộ raw keypoint hay manifest batch tạm.
ARTIFACTS.mkdir(parents=True, exist_ok=True)
report_names = ['train_evaluation.json', 'val_evaluation.json'] + (['test_evaluation.json'] if RUN_TEST else [])
payload_names = ['run_config.json', 'code_revision.txt', 'gesture_templates.yaml', 'val_diagnostics.jsonl'] + report_names + (['validation.json'] if RUN_TEST else [])
for name in payload_names:
    assert (LOCAL_ROOT / name).is_file(), f'Thiếu report bắt buộc: {name}'
completion = {
    'status': 'complete',
    'run_mode': RUN_MODE,
    'run_config_sha256': hashlib.sha256((LOCAL_ROOT / 'run_config.json').read_bytes()).hexdigest(),
    'graph_counts': {split: len(rows_by_split[split]) for split in ACTIVE_SPLITS},
    'payload_sha256': {name: hashlib.sha256((LOCAL_ROOT / name).read_bytes()).hexdigest() for name in payload_names},
}
(LOCAL_ROOT / 'completed_run.json').write_text(json.dumps(completion, indent=2), encoding='utf-8')
# Publish payload trước; marker complete bị loại khỏi rsync và chỉ được rename nguyên tử ở bước cuối.
run(['rsync', '-a', '--exclude', 'completed_run.json', '--exclude', 'data/keypoints/', '--exclude', 'data/batches/', str(LOCAL_ROOT) + '/', str(ARTIFACTS) + '/'])
for name, expected_hash in completion['payload_sha256'].items():
    drive_file = ARTIFACTS / name
    assert drive_file.is_file(), f'Drive thiếu payload: {drive_file}'
    assert hashlib.sha256(drive_file.read_bytes()).hexdigest() == expected_hash, f'Hash Drive không khớp: {drive_file}'
completion_tmp = ARTIFACTS / '.completed_run.json.tmp'
shutil.copy2(LOCAL_ROOT / 'completed_run.json', completion_tmp)
completion_tmp.replace(ARTIFACTS / 'completed_run.json')
print('Đường dẫn output chính xác:')
for path in sorted((ARTIFACTS).rglob('*')):
    if path.is_file(): print(path)

## Xử lý lỗi thường gặp

- `REPO_REF` phải trỏ đúng branch/commit có schema và ngưỡng V2; notebook dừng nếu YAML không có `version: 2` hoặc thiếu script.
- Nếu thiếu keypoint, kiểm tra `DRIVE_DATA_ROOT/keypoints/` và đường dẫn trong `train.txt`, `val.txt`, `test.txt`; bước kiểm tra trước khi chạy sẽ nêu file đầu tiên bị thiếu.
- Nếu graph đã tồn tại nhưng provenance/threshold khác, đặt `OVERWRITE=True` để dựng lại trên SSD Colab; Drive raw không bị xóa.
- Nếu validator báo thiếu graph, xem log precompute và `validation.json`; sửa dữ liệu/đường dẫn rồi chạy lại, không bỏ qua lỗi.
- Mỗi batch graph được checkpoint ngay lên Drive và ghi `graph_progress.json`, nên Colab ngắt giữa chừng có thể chạy lại với `OVERWRITE=False`.
- Kết quả nằm riêng theo chế độ và attempt ở `DRIVE_DATA_ROOT/runs/RUN_NAME/train_val/<config-hash>/` hoặc `.../full_with_test/<config-hash>/`; `completed_run.json` chỉ được publish sau cùng khi đủ report bắt buộc.